# A2 Q4 - Serving and Scale Analysis

Measures the **served two-stage pipeline** on one machine: index memory, p99
single-request latency, a back-of-envelope cost per 1,000 queries at a target
SLA, and what breaks first at 10x.

The system timed here is A2 Q1/Q2's: BM25 and frozen-embedding retrieval
scoring each impression's `article_ids_inview`, feeding the LightGBM
re-ranker. That is the path Q2 evaluated and Q5 submits. Q3's NRMS is a
*baseline* for comparison, not the served system, and is not timed.

Q4 is deliberately a single-machine measurement: index memory, p99 latency and
a cost/QPS figure only compose into an argument if they describe the same host,
so the machine spec is captured into the output rather than assumed.

Outputs `data/processed/serving_metrics.json` and `serving_benchmark.png`.
Scope a run with `SERVING_DATASETS`; see `serving_benchmark.py` for the
one-command wrapper.

## Setup and machine specification

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import ctypes
import json
import os
import platform
import statistics
import sys
import time

import lightgbm as lgb
import numpy as np
import polars as pl

from cs4406m26_assignment1c1.bm25 import BM25Index, build_index, get_scores, tokenize, top_k
from cs4406m26_assignment1c1.embeddings import batched_top_k, cosine_similarity_subset, mean_pool, normalize_rows
from cs4406m26_assignment1c1.reranker import FEATURE_COLUMNS, feature_matrix
from cs4406m26_assignment1c1.retrieval import RECENT_N_CLICKS, build_stage1_scorers


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
PROGRESS_LOG = ROOT / "build_progress.log"

# Q4 asks for one machine's measurements: index memory, p99 latency and a
# cost/QPS figure only compose into an argument if they describe the same
# host. Captured here so the design note can state it rather than imply it.
def machine_spec() -> dict:
    total_ram = None
    try:  # Windows
        class MEMORYSTATUSEX(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
        stat = MEMORYSTATUSEX()
        stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat))
        total_ram = stat.ullTotalPhys
    except Exception:  # noqa: BLE001
        try:
            total_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
        except Exception:  # noqa: BLE001
            pass
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": sys.version.split()[0],
        "logical_cores": os.cpu_count(),
        "total_ram_bytes": total_ram,
        "total_ram_gb": round(total_ram / 1024**3, 1) if total_ram else None,
        "numpy": np.__version__,
        "polars": pl.__version__,
        "lightgbm": lgb.__version__,
    }


MACHINE = machine_spec()

# Reps per timed stage. 1,000 is enough for a stable p99: the 99th percentile
# of 1,000 samples is the 10th-largest, so it is an order statistic with real
# support rather than a single outlier, while staying inside a few minutes for
# the whole matrix of (dataset x stage).
LATENCY_REPS = 1_000
WARMUP_REPS = 50          # BLAS threadpool spin-up and page-in, excluded
TOPK_MAX = 200            # Q2/Q3's corpus-wide candidate-generation K
SLA_P99_MS = 100.0        # the assignment's own example target
SEED = 0

# A plug-in price, stated as such: approximately the AWS c5.large on-demand
# rate in us-east-1 (~$0.085/h for 2 vCPUs), from general knowledge and not
# re-verified -- list prices change and vary by region. The measured quantity
# is CPU-seconds per 1,000 queries; the dollar figure is that times whatever
# price the reader prefers, and scales linearly with it (SPEC.md A2 Q4 #4).
USD_PER_VCPU_HOUR = 0.0425

BUILD_LARGE_ONLY = True
_DEFAULT_DATASETS = (["ebnerd_large", "mind_large"] if BUILD_LARGE_ONLY
                     else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"])
_env = os.environ.get("SERVING_DATASETS")
DATASETS = _env.split(",") if _env else _DEFAULT_DATASETS


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] serving_benchmark: {message}\n")
        f.flush()


log_progress(f"serving_benchmark started (datasets={DATASETS})")
print(json.dumps(MACHINE, indent=1))

{
 "platform": "Windows-11-10.0.26200-SP0",
 "processor": "Intel64 Family 6 Model 191 Stepping 2, GenuineIntel",
 "python": "3.14.6",
 "logical_cores": 20,
 "total_ram_bytes": 16833372160,
 "total_ram_gb": 15.7,
 "numpy": "2.5.1",
 "polars": "1.43.2",
 "lightgbm": "4.7.0"
}


In [2]:
def test_setup() -> None:
    assert MACHINE["total_ram_bytes"], "could not read total RAM; the scaling section needs it"
    assert MACHINE["logical_cores"] and MACHINE["logical_cores"] > 0
    for name in DATASETS:
        for f in ["articles.parquet", "article_embeddings.parquet", "history.parquet",
                  "reranker_eval_test.parquet", f"reranker_model_{name}.txt"]:
            assert (DATA_DIR / name / f).exists(), f"{name}: missing {f}"
    # Q4 measures the served two-stage pipeline, so it must use the same
    # history window Q2 trained and scored with -- a different window would
    # report the latency of a system we never evaluated.
    assert RECENT_N_CLICKS == 20, RECENT_N_CLICKS
    assert len(FEATURE_COLUMNS) == 15, len(FEATURE_COLUMNS)
    assert LATENCY_REPS >= 1000, "p99 of fewer than 1,000 samples is a single order statistic"


test_setup()
print("setup OK:", MACHINE["total_ram_gb"], "GB RAM,", MACHINE["logical_cores"], "logical cores")

setup OK: 15.7 GB RAM, 20 logical cores


## 1. Index memory

Byte-accounted rather than estimated. `sys.getsizeof` on a dict returns the
table only, not the keys, values, or the numpy buffers those values point at -
which for `BM25Index.postings` is essentially all of it. `deep_nbytes` walks
containers and adds `.nbytes`, and de-duplicates shared buffers so the
125,541 views in `embedding_lookup` are not counted as 125,541 matrices.

The feature table is measured on a bounded slice and extrapolated. Reading
`reranker_features.parquet` whole (3.2GB for `ebnerd_large`) to measure it
would reproduce the memory exhaustion this section exists to characterise.

In [3]:
def deep_nbytes(obj, _seen=None) -> int:
    """Recursive byte footprint, counting numpy buffers by `.nbytes`.

    `sys.getsizeof` on a dict returns only the table, not the keys, values or
    the numpy buffers those values point at -- which for `BM25Index.postings`
    is essentially all of it. This walks containers and adds `.nbytes` for
    arrays, so the reported index size is the thing that actually has to be
    resident rather than the size of the handle to it.

    `_seen` guards against double-counting shared buffers: `embedding_lookup`
    holds views into one matrix block, and counting each view's `.nbytes`
    would report the matrix once per article.
    """
    if _seen is None:
        _seen = set()
    marker = id(obj)
    if marker in _seen:
        return 0
    _seen.add(marker)

    if isinstance(obj, np.ndarray):
        # A view's .base owns the buffer; count the base once.
        if obj.base is not None:
            return deep_nbytes(obj.base, _seen)
        return obj.nbytes
    total = sys.getsizeof(obj)
    if isinstance(obj, dict):
        for key, value in obj.items():
            total += deep_nbytes(key, _seen) + deep_nbytes(value, _seen)
    elif isinstance(obj, (list, tuple, set, frozenset)):
        for item in obj:
            total += deep_nbytes(item, _seen)
    return total


def bm25_index_footprint(index: BM25Index) -> dict:
    """Per-field breakdown, because the fields scale differently with the
    catalogue and the 10x argument needs to know which one dominates."""
    postings_arrays = sum(a.nbytes + b.nbytes for a, b in index.postings.values())
    return {
        "doc_ids": deep_nbytes(index.doc_ids),
        "doc_len": index.doc_len.nbytes,
        "doc_norm": index.doc_norm.nbytes,
        "idf_dict": deep_nbytes(index.idf),
        "postings_container": deep_nbytes(index.postings) - postings_arrays,
        "postings_arrays": postings_arrays,
        "n_docs": index.n_docs,
        "n_terms": len(index.postings),
        "n_postings": int(sum(a.size for a, _ in index.postings.values())),
    }


def feature_store_footprint(dataset: str, sample_rows: int = 200_000) -> dict:
    """On-disk size for every persisted artifact, plus an in-memory estimate
    for the feature table measured on a bounded slice and extrapolated.

    The slice is the point: `reranker_features.parquet` is 3.2GB on disk for
    `ebnerd_large`, and reading it whole to measure it would reproduce the
    memory exhaustion this section exists to characterise.
    """
    d = DATA_DIR / dataset
    on_disk = {}
    for name in ["articles.parquet", "behaviors.parquet", "history.parquet",
                 "article_embeddings.parquet", "reranker_features.parquet",
                 "bm25_topk.parquet", "embedding_topk.parquet",
                 f"reranker_model_{dataset}.txt"]:
        p = d / name
        if p.exists():
            on_disk[name] = p.stat().st_size

    features_path = d / "reranker_features.parquet"
    per_row = None
    total_rows = None
    if features_path.exists():
        total_rows = pl.scan_parquet(features_path).select(pl.len()).collect().item()
        sample = pl.scan_parquet(features_path).head(sample_rows).collect()
        per_row = sample.estimated_size() / sample.height
        del sample
    return {
        "on_disk_bytes": on_disk,
        "on_disk_total_bytes": sum(on_disk.values()),
        "feature_rows": total_rows,
        "feature_bytes_per_row": per_row,
        "feature_in_memory_bytes": int(per_row * total_rows) if per_row else None,
    }


memory_report = {}
for name in DATASETS:
    log_progress(f"  {name}: measuring index memory")
    articles = pl.read_parquet(DATA_DIR / name / "articles.parquet",
                              columns=["article_id", "title", "abstract"])
    texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
    t0 = time.perf_counter()
    index = build_index(articles["article_id"].to_list(), [tokenize(t) for t in texts])
    build_seconds = time.perf_counter() - t0

    emb = pl.read_parquet(DATA_DIR / name / "article_embeddings.parquet")
    dim = len(emb["embedding"][0])
    matrix = emb["embedding"].list.to_array(dim).to_numpy().astype(np.float32)
    corpus_unit = normalize_rows(matrix)
    booster = lgb.Booster(model_file=str(DATA_DIR / name / f"reranker_model_{name}.txt"))

    bm25_bytes = bm25_index_footprint(index)
    memory_report[name] = {
        "bm25_index": bm25_bytes,
        "bm25_index_total_bytes": sum(v for k, v in bm25_bytes.items() if k.endswith("bytes") or
                                      k in {"doc_ids", "doc_len", "doc_norm", "idf_dict",
                                            "postings_container", "postings_arrays"}),
        "bm25_build_seconds": round(build_seconds, 2),
        "embedding_matrix_bytes": int(matrix.nbytes),
        "embedding_normalized_copy_bytes": int(corpus_unit.nbytes),
        "embedding_dim": dim,
        "reranker_model_bytes": (DATA_DIR / name / f"reranker_model_{name}.txt").stat().st_size,
        "reranker_trees": booster.num_trees(),
        "feature_store": feature_store_footprint(name),
    }
    del emb, matrix, corpus_unit, booster, index, articles, texts

for name, rep in memory_report.items():
    fs = rep["feature_store"]
    print(f"=== {name}")
    print(f"  BM25 index          {rep['bm25_index_total_bytes'] / 1024**2:8.1f} MB "
          f"({rep['bm25_index']['n_docs']:,} docs, {rep['bm25_index']['n_terms']:,} terms, "
          f"{rep['bm25_index']['n_postings']:,} postings, built in {rep['bm25_build_seconds']}s)")
    print(f"    postings arrays   {rep['bm25_index']['postings_arrays'] / 1024**2:8.1f} MB")
    print(f"    idf + containers  {(rep['bm25_index']['idf_dict'] + rep['bm25_index']['postings_container']) / 1024**2:8.1f} MB")
    print(f"  embedding matrix    {rep['embedding_matrix_bytes'] / 1024**2:8.1f} MB (dim {rep['embedding_dim']})")
    print(f"  + normalized copy   {rep['embedding_normalized_copy_bytes'] / 1024**2:8.1f} MB")
    print(f"  reranker booster    {rep['reranker_model_bytes'] / 1024**2:8.1f} MB ({rep['reranker_trees']} trees)")
    print(f"  feature store       {fs['on_disk_total_bytes'] / 1024**3:8.2f} GB on disk; "
          f"{fs['feature_rows']:,} feature rows at {fs['feature_bytes_per_row']:.0f} B/row "
          f"= {fs['feature_in_memory_bytes'] / 1024**3:.2f} GB if fully resident")

=== mind_large
  BM25 index              85.2 MB (104,151 docs, 75,091 terms, 3,864,307 postings, built in 2.57s)
    postings arrays       59.0 MB
    idf + containers      17.4 MB
  embedding matrix       305.1 MB (dim 768)
  + normalized copy      305.1 MB
  reranker booster         0.3 MB (45 trees)
  feature store           4.00 GB on disk; 97,592,931 feature rows at 111 B/row = 10.11 GB if fully resident


In [4]:
def test_memory() -> None:
    for name, rep in memory_report.items():
        bm = rep["bm25_index"]
        assert bm["n_docs"] > 0 and bm["n_terms"] > 0 and bm["n_postings"] > bm["n_docs"]
        assert rep["bm25_index_total_bytes"] > bm["postings_arrays"], "total must include the arrays"
        # The normalized copy is a separate allocation of the same shape, and
        # both are resident during scoring. Equal size is the check that
        # normalize_rows did not silently downcast or alias.
        assert rep["embedding_normalized_copy_bytes"] == rep["embedding_matrix_bytes"]
        assert rep["embedding_dim"] == 768, rep["embedding_dim"]
        fs = rep["feature_store"]
        assert fs["feature_rows"] and fs["feature_bytes_per_row"] > 0
        assert fs["on_disk_total_bytes"] > fs["on_disk_bytes"][f"reranker_model_{name}.txt"]

    # deep_nbytes must count numpy buffers, not just handles -- the whole
    # memory section is wrong if it reports dict table sizes.
    probe = {"a": np.zeros(1000, dtype=np.float64)}          # 8,000 bytes of payload
    assert deep_nbytes(probe) > 8000, deep_nbytes(probe)
    assert sys.getsizeof(probe) < 1000, "getsizeof would have missed the buffer"
    # Views must not be double-counted. deep_nbytes redirects a view to its
    # `.base`, which owns the buffer -- without the `_seen` guard that would
    # report the whole block once per view, so emb_lookup's 125,541 views
    # would come out as 125,541 copies of the matrix. The probe block has to
    # be large enough that its payload dominates the dict and key overhead;
    # a (100, 8) block is only 3.2KB against ~7.5KB of container, which made
    # an earlier version of this assertion fail on correct code.
    block = np.zeros((100, 1000), dtype=np.float32)   # 400,000 bytes
    views = {i: block[i] for i in range(100)}
    counted = deep_nbytes(views)
    assert counted < block.nbytes * 2, f"views double-counted: {counted:,} vs {block.nbytes:,}"
    assert counted > block.nbytes, f"buffer not counted at all: {counted:,}"


test_memory()
print("memory accounting OK (buffers counted, views not double-counted)")

memory accounting OK (buffers counted, views not double-counted)


## 2. p99 latency for a single request

Every stage of one request is timed separately and end to end, over
`LATENCY_REPS` sampled requests from Q2's scored test population, after
`WARMUP_REPS` untimed requests to absorb BLAS thread-pool spin-up and
page-in.

Index construction, embedding normalization and the feature-store load are
**excluded**: they are startup costs, and charging them to a request would
report a latency no served system has. They appear in section 1 instead.

Two candidate paths are reported because Q4's wording ("candidate generation +
re-ranking") and the served system differ, which `SPEC.md` Q4 §1 already
records: the served path scores `article_ids_inview`, while corpus-wide
top-200 retrieval is what "candidate generation" names. Both are measured.

In [5]:
def percentiles(samples_ms) -> dict:
    a = np.sort(np.asarray(samples_ms, dtype=np.float64))
    return {
        "n": int(a.size),
        "mean_ms": float(a.mean()),
        "p50_ms": float(np.percentile(a, 50)),
        "p95_ms": float(np.percentile(a, 95)),
        "p99_ms": float(np.percentile(a, 99)),
        "max_ms": float(a.max()),
    }


def build_serving_state(dataset: str) -> dict:
    """Everything a warm server would already hold, built once.

    Deliberately excluded from the timings below: index construction,
    embedding normalization and the feature-store load are startup costs, not
    per-request costs, and charging them to a request would report a latency
    no served system has. They are reported separately in the memory section.
    """
    articles = pl.read_parquet(DATA_DIR / dataset / "articles.parquet",
                              columns=["article_id", "title", "abstract"])
    scorers = build_stage1_scorers(
        articles, DATA_DIR / dataset / "history.parquet",
        DATA_DIR / dataset / "article_embeddings.parquet")

    # Rebuilt here rather than reaching into build_stage1_scorers' closure,
    # because the corpus-wide top-K path (candidate generation) needs the
    # index and matrix as values, and the served path needs the adapters.
    texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
    index = build_index(articles["article_id"].to_list(), [tokenize(t) for t in texts])
    title_lookup = dict(zip(articles["article_id"].to_list(), articles["title"].to_list()))
    emb = pl.read_parquet(DATA_DIR / dataset / "article_embeddings.parquet")
    dim = len(emb["embedding"][0])
    emb_mat = emb["embedding"].list.to_array(dim).to_numpy().astype(np.float32)
    emb_pos = {a: i for i, a in enumerate(emb["article_id"].to_list())}
    doc_ids = articles["article_id"].to_numpy()
    matrix = emb_mat[np.array([emb_pos[a] for a in doc_ids])]
    corpus_unit = normalize_rows(matrix)
    emb_lookup = {a: matrix[i] for i, a in enumerate(doc_ids)}
    id_to_idx = {a: i for i, a in enumerate(doc_ids)}
    del emb, emb_mat, emb_pos

    history = dict(zip(*(
        pl.scan_parquet(DATA_DIR / dataset / "history.parquet")
        .select("user_id", pl.col("article_id_sequence").list.tail(RECENT_N_CLICKS))
        .collect().to_dict(as_series=False).values())))

    # Warm feature store: A2 Q2's scored test population, sorted by
    # impression_id so a request is a contiguous slice. A real deployment
    # would serve these from a KV store; the cost that matters per request is
    # gathering `n_candidates` rows and casting them, which a slice measures
    # honestly. A per-request predicate over the whole frame would instead
    # measure a full scan, which no feature store does.
    feats = (pl.read_parquet(DATA_DIR / dataset / "reranker_eval_test.parquet")
             .sort(["impression_id", "article_id"]))
    lengths = feats.group_by("impression_id", maintain_order=True).len()["len"].to_numpy()
    offsets = np.concatenate([[0], np.cumsum(lengths, dtype=np.int64)])
    imp_order = feats["impression_id"].unique(maintain_order=True).to_list()
    slice_of = {imp: (int(offsets[i]), int(lengths[i])) for i, imp in enumerate(imp_order)}

    booster = lgb.Booster(model_file=str(DATA_DIR / dataset / f"reranker_model_{dataset}.txt"))

    rng = np.random.default_rng(SEED)
    pick = rng.choice(len(imp_order), size=min(LATENCY_REPS + WARMUP_REPS, len(imp_order)),
                      replace=False)
    requests = []
    uid_by_imp = dict(zip(feats["impression_id"].to_list(), feats["user_id"].to_list()))
    for i in pick:
        imp = imp_order[i]
        start, n = slice_of[imp]
        requests.append({
            "impression_id": imp,
            "user_id": uid_by_imp[imp],
            "article_ids": feats["article_id"].slice(start, n).to_list(),
            "start": start, "n": n,
        })

    log_progress(f"  {dataset}: serving state ready, {len(requests)} sampled requests")
    return {"index": index, "title_lookup": title_lookup, "doc_ids": doc_ids,
            "corpus_unit": corpus_unit, "emb_lookup": emb_lookup, "id_to_idx": id_to_idx,
            "history": history, "feats": feats, "booster": booster,
            "scorers": scorers, "requests": requests,
            "bm25_id_to_idx": {a: i for i, a in enumerate(index.doc_ids)}}


def measure_latency(dataset: str, state: dict) -> dict:
    idx, requests = state["index"], state["requests"]
    feats, booster = state["feats"], state["booster"]
    bm25_pos, id_to_idx = state["bm25_id_to_idx"], state["id_to_idx"]
    hist, titles, emb_lookup = state["history"], state["title_lookup"], state["emb_lookup"]
    corpus_unit, doc_ids = state["corpus_unit"], state["doc_ids"]

    stages = {k: [] for k in [
        "query_build", "bm25_inview", "embedding_inview", "feature_assembly",
        "reranker_predict", "served_end_to_end", "candgen_bm25_top200",
        "candgen_embedding_top200", "candgen_embedding_top200_batchfn"]}
    n_docs = corpus_unit.shape[0]
    cand_counts = []

    for rep, req in enumerate(requests):
        ids = req["article_ids"]
        seq = list(hist.get(req["user_id"], ()))[-RECENT_N_CLICKS:]
        timed = rep >= WARMUP_REPS  # warm-up excluded: BLAS spin-up, page-in

        t0 = time.perf_counter()
        tokens = tokenize(" ".join(t for t in (titles.get(a, "") for a in seq) if t))
        query_vec = mean_pool(seq, emb_lookup)
        t1 = time.perf_counter()

        scores = get_scores(idx, tokens)
        bm25_scored = {a: float(scores[bm25_pos[a]]) for a in ids}
        t2 = time.perf_counter()

        emb_scored = cosine_similarity_subset(query_vec, corpus_unit, doc_ids, id_to_idx, ids)
        t3 = time.perf_counter()

        chunk = feats.slice(req["start"], req["n"])
        X = feature_matrix(chunk)
        # Stage 1's fresh scores replace the persisted ones: at serving time
        # these two columns are computed by the calls just timed, not read.
        X[:, FEATURE_COLUMNS.index("bm25_score")] = [bm25_scored[a] for a in ids]
        X[:, FEATURE_COLUMNS.index("embedding_score")] = [emb_scored.get(a, 0.0) for a in ids]
        t4 = time.perf_counter()

        booster.predict(X)
        t5 = time.perf_counter()

        if timed:
            stages["query_build"].append((t1 - t0) * 1e3)
            stages["bm25_inview"].append((t2 - t1) * 1e3)
            stages["embedding_inview"].append((t3 - t2) * 1e3)
            stages["feature_assembly"].append((t4 - t3) * 1e3)
            stages["reranker_predict"].append((t5 - t4) * 1e3)
            stages["served_end_to_end"].append((t5 - t0) * 1e3)
            cand_counts.append(len(ids))

        # Corpus-wide candidate generation, timed separately: Q4's wording is
        # "candidate generation + re-ranking", and A1 Q4 #1 records why the
        # served path scores article_ids_inview instead. Both are reported.
        t6 = time.perf_counter()
        top_k(idx, tokens, TOPK_MAX)
        t7 = time.perf_counter()
        # Serving-shaped embedding candidate generation: the unit corpus is
        # precomputed once (it is part of the resident index measured in
        # section 1), so a request is one matvec plus a partial sort.
        if query_vec is not None:
            q = query_vec.astype(np.float32)
            q /= (np.linalg.norm(q) or 1.0)
            sims = corpus_unit @ q
            part = np.argpartition(-sims, TOPK_MAX)[:TOPK_MAX] if TOPK_MAX < n_docs else np.arange(n_docs)
            part[np.argsort(-sims[part])]
        t8 = time.perf_counter()
        # The offline batch function called per request -- deliberately
        # measured as the anti-pattern it is. batched_top_k normalizes the
        # whole corpus inside every call (it amortizes that over 2,000
        # queries in Q3's offline pass), so per-request use re-normalizes
        # n_docs x 768 floats for one query. On the demo track this was
        # 34ms against a 1.6ms served path; at ebnerd_large's 125,541 docs
        # it would falsely show the system breaching a 100ms SLA at 1x.
        if query_vec is not None:
            batched_top_k(query_vec[None, :], corpus_unit, doc_ids, TOPK_MAX)
        t9 = time.perf_counter()
        if timed:
            stages["candgen_bm25_top200"].append((t7 - t6) * 1e3)
            stages["candgen_embedding_top200"].append((t8 - t7) * 1e3)
            stages["candgen_embedding_top200_batchfn"].append((t9 - t8) * 1e3)

        if (rep + 1) % 250 == 0:
            log_progress(f"  {dataset}: {rep + 1}/{len(requests)} requests timed")

    out = {k: percentiles(v) for k, v in stages.items() if v}
    out["candidates_per_request"] = {
        "mean": float(np.mean(cand_counts)), "p50": float(np.percentile(cand_counts, 50)),
        "p99": float(np.percentile(cand_counts, 99)), "max": int(np.max(cand_counts))}
    return out


latency_report = {}
for name in DATASETS:
    state = build_serving_state(name)
    latency_report[name] = measure_latency(name, state)
    del state

for name, rep in latency_report.items():
    print(f"=== {name}   ({rep['candidates_per_request']['mean']:.1f} candidates/request mean, "
          f"{rep['candidates_per_request']['max']} max)")
    print(f"  {'stage':28s} {'mean':>8s} {'p50':>8s} {'p95':>8s} {'p99':>8s}")
    for stage in ["query_build", "bm25_inview", "embedding_inview", "feature_assembly",
                  "reranker_predict", "served_end_to_end", "candgen_bm25_top200",
                  "candgen_embedding_top200", "candgen_embedding_top200_batchfn"]:
        s = rep[stage]
        print(f"  {stage:28s} {s['mean_ms']:8.3f} {s['p50_ms']:8.3f} {s['p95_ms']:8.3f} {s['p99_ms']:8.3f}")

=== mind_large   (34.0 candidates/request mean, 254 max)
  stage                            mean      p50      p95      p99
  query_build                     0.156    0.168    0.217    0.263
  bm25_inview                     5.441    6.114    7.697    9.481
  embedding_inview                0.092    0.075    0.199    0.264
  feature_assembly                0.170    0.165    0.213    0.271
  reranker_predict                0.340    0.342    0.397    0.422
  served_end_to_end               6.199    6.894    8.537   10.426
  candgen_bm25_top200             6.807    7.157   12.532   14.561
  candgen_embedding_top200        6.097    6.269    6.668    6.891
  candgen_embedding_top200_batchfn  241.393  246.362  263.576  274.347


In [6]:
def test_latency() -> None:
    for name, rep in latency_report.items():
        served = rep["served_end_to_end"]
        assert served["n"] == LATENCY_REPS, (name, served["n"])
        # Percentiles must be ordered, or the sort/percentile call is wrong.
        assert served["p50_ms"] <= served["p95_ms"] <= served["p99_ms"] <= served["max_ms"]
        # The end-to-end figure is the sum of its parts, so it cannot be
        # smaller than the largest one. A violation would mean the timers
        # overlap or a stage was measured outside the end-to-end window.
        parts = ["query_build", "bm25_inview", "embedding_inview",
                 "feature_assembly", "reranker_predict"]
        assert served["mean_ms"] >= max(rep[p]["mean_ms"] for p in parts)
        assert abs(served["mean_ms"] - sum(rep[p]["mean_ms"] for p in parts)) < 0.5 * served["mean_ms"], \
            "end-to-end differs from the sum of stages by more than 50% -- timers are not contiguous"
        for stage in parts:
            assert rep[stage]["mean_ms"] > 0, f"{name}/{stage} measured as zero"
        # Corpus-wide candidate generation must cost more than scoring the
        # in-view set: it scores the whole catalogue and then selects top-200.
        assert rep["candgen_bm25_top200"]["mean_ms"] >= rep["bm25_inview"]["mean_ms"] * 0.5
        # The serving-shaped top-K must beat the offline batch function called
        # per request, or the anti-pattern is not one and the note is wrong.
        assert rep["candgen_embedding_top200"]["mean_ms"] < rep["candgen_embedding_top200_batchfn"]["mean_ms"],             (rep["candgen_embedding_top200"]["mean_ms"], rep["candgen_embedding_top200_batchfn"]["mean_ms"])
        c = rep["candidates_per_request"]
        assert 1 <= c["p50"] <= c["max"]


test_latency()
print("latency OK (percentiles ordered, end-to-end reconciles with its stages)")

latency OK (percentiles ordered, end-to-end reconciles with its stages)


## 3. Cost and QPS at the SLA

Throughput comes from the **mean** service time and the SLA check from the
**p99** - a queue's service rate is set by the mean, while the tail is what
breaches. The price per vCPU-hour is an input assumption, not a finding, and
is recorded in the output alongside the figure it produces.

In [7]:
def cost_model(dataset: str) -> dict:
    """Throughput and cost per 1,000 queries, from the measured served path.

    Throughput is derived from the **mean**, not the p99: a queue's service
    rate is set by mean service time, while the p99 is what the SLA is
    checked against. Using p99 for both would understate capacity by the
    width of the tail.

    Stated assumption, not a finding: one request occupies one process, and
    processes scale linearly across vCPUs. Both are optimistic -- numpy's
    BLAS may already use several threads inside a single request (so a
    "core" here is not cleanly one core), and real deployments lose headroom
    to queueing well before 100% utilisation. The figure is a
    back-of-envelope, which is what the assignment asks for.
    """
    served = latency_report[dataset]["served_end_to_end"]
    qps_per_process = 1000.0 / served["mean_ms"]
    meets_sla = served["p99_ms"] < SLA_P99_MS
    # Little's law at the SLA: with p99 as the latency budget, a single
    # process can hold at most budget/mean requests in flight before the
    # tail breaches. Reported as headroom rather than used as capacity.
    headroom = SLA_P99_MS / served["p99_ms"]
    cpu_seconds_per_1k = 1000.0 / qps_per_process
    return {
        "served_mean_ms": served["mean_ms"],
        "served_p99_ms": served["p99_ms"],
        "sla_p99_ms": SLA_P99_MS,
        "meets_sla": bool(meets_sla),
        "p99_headroom_x": round(headroom, 2),
        "qps_per_process": round(qps_per_process, 1),
        "qps_per_host_optimistic": round(qps_per_process * MACHINE["logical_cores"], 1),
        "cpu_seconds_per_1000_queries": round(cpu_seconds_per_1k, 2),
        "usd_per_vcpu_hour": USD_PER_VCPU_HOUR,
        "usd_per_1000_queries": round(cpu_seconds_per_1k / 3600.0 * USD_PER_VCPU_HOUR, 6),
        "processes_for_1000_qps": int(np.ceil(1000.0 / qps_per_process)),
    }


cost_report = {name: cost_model(name) for name in DATASETS}
for name, c in cost_report.items():
    print(f"=== {name}")
    print(f"  served mean {c['served_mean_ms']:.2f} ms, p99 {c['served_p99_ms']:.2f} ms "
          f"vs SLA {c['sla_p99_ms']:.0f} ms -> {'MEETS' if c['meets_sla'] else 'BREACHES'} "
          f"({c['p99_headroom_x']}x headroom)")
    print(f"  {c['qps_per_process']:.1f} QPS/process; {c['qps_per_host_optimistic']:.1f} QPS "
          f"across {MACHINE['logical_cores']} logical cores (optimistic, linear)")
    print(f"  {c['cpu_seconds_per_1000_queries']:.2f} CPU-s per 1,000 queries "
          f"-> ${c['usd_per_1000_queries']:.6f} at ${USD_PER_VCPU_HOUR}/vCPU-hour")
    print(f"  {c['processes_for_1000_qps']} processes to sustain 1,000 QPS")

=== mind_large
  served mean 6.20 ms, p99 10.43 ms vs SLA 100 ms -> MEETS (9.59x headroom)
  161.3 QPS/process; 3226.5 QPS across 20 logical cores (optimistic, linear)
  6.20 CPU-s per 1,000 queries -> $0.000073 at $0.0425/vCPU-hour
  7 processes to sustain 1,000 QPS


In [8]:
def test_cost() -> None:
    for name, c in cost_report.items():
        served = latency_report[name]["served_end_to_end"]
        # Throughput must come from the mean, not the p99.
        assert abs(c["qps_per_process"] - 1000.0 / served["mean_ms"]) < 0.1
        assert c["meets_sla"] == (served["p99_ms"] < SLA_P99_MS)
        # Cost is CPU-seconds times a price; both directions must reconcile.
        assert abs(c["cpu_seconds_per_1000_queries"] - 1000.0 / c["qps_per_process"]) < 0.05
        # Recomputed from the unrounded throughput, not the 2-dp CPU-seconds,
        # and compared at the 6-dp granularity the USD figure is stored at:
        # a 1e-9 tolerance against a value rounded to 1e-6 fails on correct
        # arithmetic.
        expected = (1000.0 / (1000.0 / served["mean_ms"])) / 3600.0 * USD_PER_VCPU_HOUR
        assert abs(c["usd_per_1000_queries"] - expected) < 1e-6, (c["usd_per_1000_queries"], expected)
        assert c["processes_for_1000_qps"] >= 1


test_cost()
print("cost model OK (throughput from the mean, SLA from the p99, arithmetic reconciles)")

cost model OK (throughput from the mean, SLA from the p99, arithmetic reconciles)


## 4. What breaks first at 10x

Arithmetic on the measured footprints, with each component labelled by how it
actually scales: the embedding matrix and BM25 postings with the **catalogue**,
the feature store with **impressions x candidates** (so 10x traffic is 10x the
store even if the catalogue is unchanged), and the re-ranker with neither - a
fixed tree count, which is why it never appears in the answer.

In [9]:
def scaling_projection(dataset: str, factors=(1, 2, 5, 10)) -> dict:
    """Which component exceeds this machine first, as arithmetic on the
    measured footprints rather than as an opinion.

    Each component is labelled by how it actually scales, because that is
    what decides the answer:

    - `embedding_matrix` and its normalized copy scale **linearly with the
      catalogue**, and both must be resident for brute-force cosine.
    - `postings_arrays` scale with total postings, i.e. catalogue x mean
      document length -- also linear here, since 10x more articles of the
      same length is 10x the postings.
    - the `feature_store` scales with **impressions x candidates**, which is
      the term that grows fastest under "10x the load": 10x traffic is 10x
      impressions over a catalogue that may not have grown at all.
    - `reranker_model` does **not** scale with data at all (a fixed 45-53
      trees), which is why it never appears in the answer.
    """
    m = memory_report[dataset]
    fs = m["feature_store"]
    resident_now = (m["bm25_index_total_bytes"] + m["embedding_matrix_bytes"]
                    + m["embedding_normalized_copy_bytes"] + m["reranker_model_bytes"])
    components = {
        "bm25_index": m["bm25_index_total_bytes"],
        "embedding_matrix_plus_copy": m["embedding_matrix_bytes"] + m["embedding_normalized_copy_bytes"],
        "reranker_model": m["reranker_model_bytes"],
    }
    ram = MACHINE["total_ram_bytes"]

    rows = []
    for f in factors:
        # The serving set scales with the catalogue; the feature store with
        # traffic. Both are multiplied by f here, which is the pessimistic
        # reading of "10x the current load" and the one worth planning for.
        serving = m["bm25_index_total_bytes"] * f + \
                  (m["embedding_matrix_bytes"] + m["embedding_normalized_copy_bytes"]) * f + \
                  m["reranker_model_bytes"]
        features = (fs["feature_in_memory_bytes"] or 0) * f
        rows.append({
            "factor": f,
            "serving_resident_bytes": int(serving),
            "serving_resident_gb": round(serving / 1024**3, 2),
            "feature_store_bytes": int(features),
            "feature_store_gb": round(features / 1024**3, 2),
            "serving_fits_in_ram": bool(serving < ram) if ram else None,
            "serving_plus_features_fits": bool(serving + features < ram) if ram else None,
            # Brute-force cosine is O(n_docs x dim) per query, so the
            # embedding half of retrieval latency scales with the catalogue.
            "projected_embedding_candgen_p99_ms": round(
                latency_report[dataset]["candgen_embedding_top200"]["p99_ms"] * f, 3),
            "projected_bm25_candgen_p99_ms": round(
                latency_report[dataset]["candgen_bm25_top200"]["p99_ms"] * f, 3),
        })

    # First factor at which the serving set alone stops fitting.
    breaks_at = next((r["factor"] for r in rows if r["serving_fits_in_ram"] is False), None)
    return {
        "resident_now_bytes": int(resident_now),
        "resident_now_gb": round(resident_now / 1024**3, 2),
        "machine_ram_gb": MACHINE["total_ram_gb"],
        "components_now_bytes": components,
        "projection": rows,
        "serving_set_exceeds_ram_at_factor": breaks_at,
        "dominant_component_now": max(components, key=components.get),
    }


scaling_report = {name: scaling_projection(name) for name in DATASETS}
for name, s in scaling_report.items():
    print(f"=== {name}   resident serving set now {s['resident_now_gb']} GB "
          f"of {s['machine_ram_gb']} GB RAM; dominant component: {s['dominant_component_now']}")
    print(f"  {'x':>3s} {'serving GB':>11s} {'features GB':>12s} {'both fit':>9s} "
          f"{'bm25 p99 ms':>12s} {'emb p99 ms':>11s}")
    for r in s["projection"]:
        print(f"  {r['factor']:3d} {r['serving_resident_gb']:11.2f} {r['feature_store_gb']:12.2f} "
              f"{str(r['serving_plus_features_fits']):>9s} "
              f"{r['projected_bm25_candgen_p99_ms']:12.2f} {r['projected_embedding_candgen_p99_ms']:11.2f}")

=== mind_large   resident serving set now 0.68 GB of 15.7 GB RAM; dominant component: embedding_matrix_plus_copy
    x  serving GB  features GB  both fit  bm25 p99 ms  emb p99 ms
    1        0.68        10.11      True        14.56        6.89
    2        1.36        20.21     False        29.12       13.78
    5        3.40        50.54     False        72.81       34.45
   10        6.79       101.07     False       145.61       68.91


In [10]:
def test_scaling() -> None:
    for name, s in scaling_report.items():
        rows = s["projection"]
        assert [r["factor"] for r in rows] == sorted(r["factor"] for r in rows)
        # Monotone in the scale factor, or the projection is not a projection.
        for a, b in zip(rows, rows[1:]):
            assert b["serving_resident_bytes"] > a["serving_resident_bytes"]
            assert b["feature_store_bytes"] >= a["feature_store_bytes"]
            assert b["projected_embedding_candgen_p99_ms"] > a["projected_embedding_candgen_p99_ms"]
        # The 1x row must reproduce the measured footprint, not a model of it.
        one = next(r for r in rows if r["factor"] == 1)
        assert one["serving_resident_bytes"] == s["resident_now_bytes"]
        # Projections are stored at 3 dp, so compare at that precision.
        raw_p99 = latency_report[name]["candgen_embedding_top200"]["p99_ms"]
        assert one["projected_embedding_candgen_p99_ms"] == round(raw_p99, 3),             (one["projected_embedding_candgen_p99_ms"], raw_p99)
        assert s["dominant_component_now"] in s["components_now_bytes"]
        # The model must not claim the reranker grows with data: it is a fixed
        # tree count, and treating it as scaling would misattribute the bound.
        assert all(r["serving_resident_bytes"]
                   - memory_report[name]["reranker_model_bytes"] > 0 for r in rows)


test_scaling()
print("scaling OK (monotone, 1x reproduces the measurement, fixed costs not scaled)")

scaling OK (monotone, 1x reproduces the measurement, fixed costs not scaled)


## Figures

In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Drawn from the persisted serving_metrics.json files, not from this kernel's
# in-memory reports. The benchmark runs one dataset per kernel
# (SERVING_DATASETS), so an in-memory plot would only ever show one dataset;
# reading the artifacts shows every dataset that has been measured, and the
# figure can be regenerated in seconds without re-running the benchmark.
# Every number in the figure is therefore the same number SPEC.md quotes.
PLOT_DATASETS = [d for d in ["ebnerd_large", "mind_large"]
                 if (DATA_DIR / d / "serving_metrics.json").exists()]
assert PLOT_DATASETS, "no serving_metrics.json found for any large dataset"
P = {d: json.loads((DATA_DIR / d / "serving_metrics.json").read_text(encoding="utf-8"))
     for d in PLOT_DATASETS}
ram_gb = P[PLOT_DATASETS[0]]["machine"]["total_ram_gb"]

BG, FG, GRID = "#121212", "#e8e8e8", "#2e2e2e"
PALETTE = ["#4db8ff", "#ffa94d", "#8ce99a", "#ff8787", "#b197fc"]
STAGES = ["query_build", "bm25_inview", "embedding_inview", "feature_assembly", "reranker_predict"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), facecolor=BG)
for ax in axes:
    ax.set_facecolor(BG)
    ax.tick_params(colors=FG, labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(GRID)
    ax.grid(True, color=GRID, linestyle="--", linewidth=0.6, alpha=0.7)
    ax.set_axisbelow(True)

# (1) Where a served request's time goes, stacked by stage, with p99 marked.
ax = axes[0]
bottoms = np.zeros(len(PLOT_DATASETS))
for j, stage in enumerate(STAGES):
    vals = np.array([P[d]["latency"][stage]["mean_ms"] for d in PLOT_DATASETS])
    ax.barh(PLOT_DATASETS, vals, left=bottoms, color=PALETTE[j], edgecolor=BG,
            label=stage.replace("_", " "))
    bottoms += vals
for i, d in enumerate(PLOT_DATASETS):
    p99 = P[d]["latency"]["served_end_to_end"]["p99_ms"]
    ax.plot([p99], [i], marker="D", color=FG, markersize=7, linestyle="none",
            label="served p99" if i == 0 else None)
    ax.text(p99 + 0.15, i, f"p99 {p99:.1f} ms", color=FG, va="center", fontsize=9)
ax.set_xlabel("ms per request (stage means, stacked)", color=FG)
ax.set_title("Served path: BM25 in-view scoring dominates", color=FG, fontsize=11)
ax.set_xlim(0, max(P[d]["latency"]["served_end_to_end"]["p99_ms"] for d in PLOT_DATASETS) * 1.35)
# Below the axes, not inside: inside, any corner covers one dataset's p99 label.
ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=FG, fontsize=8, ncol=3,
          loc="upper center", bbox_to_anchor=(0.5, -0.16)).get_frame().set_alpha(0.9)

# (2) Serving-shaped vs offline-batch-function candidate generation (log x).
ax = axes[1]
labels = ["bm25 top-200", "embedding top-200\n(serving-shaped)", "embedding top-200\n(batched_top_k per request)"]
keys = ["candgen_bm25_top200", "candgen_embedding_top200", "candgen_embedding_top200_batchfn"]
y = np.arange(len(labels))
h = 0.36
for j, d in enumerate(PLOT_DATASETS):
    vals = [P[d]["latency"][k]["p99_ms"] for k in keys]
    ax.barh(y + (j - 0.5) * h, vals, height=h, color=PALETTE[j], edgecolor=BG, label=d)
    for yy, v in zip(y + (j - 0.5) * h, vals):
        ax.text(v * 1.08, yy, f"{v:.0f}" if v >= 10 else f"{v:.1f}", color=FG, va="center", fontsize=8)
ax.axvline(100, color="#ff6b6b", linestyle=":", linewidth=1.6)
ax.text(105, len(labels) - 0.6, "100 ms SLA", color="#ff6b6b", fontsize=9)
ax.set_xscale("log")
ax.set_yticks(y)
ax.set_yticklabels(labels, color=FG, fontsize=9)
ax.set_xlabel("p99 ms per request (log)", color=FG)
ax.set_title("Corpus-wide candidate generation: API shape is 40x", color=FG, fontsize=11)
ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=FG, fontsize=8, loc="lower right").get_frame().set_alpha(0.9)

# (3) Resident memory against this machine's ceiling, by scale factor.
ax = axes[2]
for j, d in enumerate(PLOT_DATASETS):
    rows = P[d]["scaling"]["projection"]
    xs = [r["factor"] for r in rows]
    ax.plot(xs, [r["serving_resident_gb"] for r in rows], marker="o", color=PALETTE[j],
            label=f"{d}: serving set")
    ax.plot(xs, [r["feature_store_gb"] for r in rows], marker="s", linestyle="--",
            color=PALETTE[j], alpha=0.7, label=f"{d}: offline feature table")
ax.axhline(ram_gb, color="#ff6b6b", linestyle=":", linewidth=1.6)
ax.text(3.2, ram_gb * 0.62, f"this machine: {ram_gb} GB", color="#ff6b6b", fontsize=9)
ax.set_yscale("log")
ax.set_xticks([1, 2, 5, 10])
ax.set_xlabel("scale factor on catalogue and traffic", color=FG)
ax.set_ylabel("GB resident (log)", color=FG)
ax.set_title("What exceeds the machine first", color=FG, fontsize=11)
ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=FG, fontsize=8, loc="upper left").get_frame().set_alpha(0.9)

fig.suptitle(f"A2 Q4 serving benchmark - {P[PLOT_DATASETS[0]]['machine']['logical_cores']} logical cores, "
             f"{ram_gb} GB RAM, {P[PLOT_DATASETS[0]]['hyperparameters']['latency_reps']} requests per dataset",
             color=FG, fontsize=12)
fig.tight_layout(rect=(0, 0.04, 1, 0.95))
PLOT_PATH = ROOT / "serving_benchmark.png"
fig.savefig(PLOT_PATH, facecolor=BG, dpi=150)
plt.close(fig)
print("wrote", PLOT_PATH, "for", PLOT_DATASETS)

wrote C:\Users\HP\cs4406m26-assignment1c1\serving_benchmark.png for ['ebnerd_large', 'mind_large']


In [12]:
def test_plot() -> None:
    assert PLOT_PATH.exists() and PLOT_PATH.stat().st_size > 20_000, PLOT_PATH
    # The figure is drawn from the persisted artifacts, so what it shows must
    # be what the JSON says: spot-check the served p99 it annotated.
    for d in PLOT_DATASETS:
        assert P[d]["latency"]["served_end_to_end"]["p99_ms"] > 0
    # The project's plots are specified on a #121212 ground (CLAUDE.md), and a
    # default-white figure saved by accident is the easy mistake here.
    assert BG == "#121212"


test_plot()
print("plot OK")

plot OK


## Persist `serving_metrics.json`

In [13]:
def write_serving_metrics(dataset: str) -> Path:
    """One file per dataset under data/processed/{dataset}/, matching
    eval_metrics.json and reranker_eval_metrics.json, so a per-kernel run
    scoped by SERVING_DATASETS never overwrites the other dataset's result.
    The machine spec is repeated in each, because each is a self-contained
    statement about one host."""
    payload = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "machine": MACHINE,
        "hyperparameters": {
            "latency_reps": LATENCY_REPS, "warmup_reps": WARMUP_REPS,
            "topk_max": TOPK_MAX, "sla_p99_ms": SLA_P99_MS,
            "recent_n_clicks": RECENT_N_CLICKS, "seed": SEED,
            "usd_per_vcpu_hour": USD_PER_VCPU_HOUR,
        },
        "dataset": dataset,
        "index_memory": memory_report[dataset],
        "latency": latency_report[dataset],
        "cost": cost_report[dataset],
        "scaling": scaling_report[dataset],
        "notes": {
            "served_path": "history -> BM25 get_scores + embedding cosine over article_ids_inview "
                           "-> feature assembly -> LightGBM predict. This is the path A2 Q2 "
                           "evaluated and Q5 submits, so it is the one timed end to end.",
            "candgen_path": "corpus-wide top-200 via bm25.top_k and embeddings.batched_top_k, "
                            "timed separately because Q4's wording is 'candidate generation + "
                            "re-ranking' while the served path scores the in-view set "
                            "(SPEC.md Q4 #1 records why).",
            "excluded_from_latency": "index construction, embedding normalization and the "
                                     "feature-store load are startup costs, reported under "
                                     "index_memory instead of charged to a request.",
            "threading_caveat": "latency is per process, not per core: numpy's BLAS may use "
                                "several threads inside one request, so qps_per_host_optimistic "
                                "is an upper bound.",
            "nrms_not_served": "Q3's NRMS is a baseline, not the served system, and is not "
                               "timed here. Serving it locally would need the Keras weights "
                               "plus a runtime this environment does not have (Python 3.14).",
        },
    }
    path = DATA_DIR / dataset / "serving_metrics.json"
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(tmp, path)
    log_progress(f"  {dataset}: serving_metrics.json written ({path.stat().st_size:,} bytes)")
    return path


SERVING_METRICS_PATHS = {name: write_serving_metrics(name) for name in DATASETS}
for name, p in SERVING_METRICS_PATHS.items():
    print("wrote", p)

wrote C:\Users\HP\cs4406m26-assignment1c1\data\processed\mind_large\serving_metrics.json


In [14]:
def test_persist() -> None:
    for name, path in SERVING_METRICS_PATHS.items():
        payload = json.loads(path.read_text(encoding="utf-8"))
        assert payload["schema_version"] == 1 and payload["dataset"] == name
        assert payload["machine"]["total_ram_gb"] == MACHINE["total_ram_gb"]
        # Round-trip the numbers the design note will quote.
        assert abs(payload["latency"]["served_end_to_end"]["p99_ms"]
                   - latency_report[name]["served_end_to_end"]["p99_ms"]) < 1e-9
        assert abs(payload["cost"]["usd_per_1000_queries"]
                   - cost_report[name]["usd_per_1000_queries"]) < 1e-12
        # The caveats are part of the result, not decoration: a cost/QPS
        # figure without its stated assumptions is not a back-of-envelope,
        # it is a claim.
        for key in ["served_path", "candgen_path", "excluded_from_latency", "threading_caveat"]:
            assert payload["notes"][key]


test_persist()
print("serving_metrics.json OK (round-trips, caveats recorded)")

serving_metrics.json OK (round-trips, caveats recorded)


# Manual Verification Complete